# Backtesting Framework Tutorial

This tutorial demonstrates how to use QuantStrata's backtesting infrastructure to evaluate trading strategies.

**Topics covered:**
- Setting up a backtest
- Writing trading strategies
- Analyzing performance metrics
- P&L attribution
- Visualizing results

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '../../..')

import numpy as np
import matplotlib.pyplot as plt
from datetime import date, timedelta
from dataclasses import dataclass

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

np.random.seed(42)

## 1. Setup and Data Preparation

First, let's import the backtesting components and create synthetic price data.

In [ ]:
from src.backtesting.core import BacktestEngine, BacktestConfig
from src.backtesting.core.metrics import compute_all_metrics, compute_sharpe_ratio, compute_max_drawdown
from src.backtesting.data import DictDataProvider, SimpleMarketSnapshot

# Generate 2 years of daily data for 3 stocks
n_days = 504  # ~2 years
dates = [date(2022, 1, 3) + timedelta(days=i) for i in range(n_days)]

# Simulate GBM paths
def simulate_gbm(s0, mu, sigma, n):
    dt = 1/252
    returns = np.random.normal(mu * dt, sigma * np.sqrt(dt), n)
    return s0 * np.cumprod(1 + returns)

# Create price series
aapl = simulate_gbm(150, 0.10, 0.25, n_days)
msft = simulate_gbm(300, 0.12, 0.22, n_days)
googl = simulate_gbm(2800, 0.08, 0.28, n_days)

# Build data provider
data = {}
for i, d in enumerate(dates):
    data[d] = {"AAPL": aapl[i], "MSFT": msft[i], "GOOGL": googl[i]}

provider = DictDataProvider(data)

print(f"Data range: {provider.start_date} to {provider.end_date}")
print(f"Number of dates: {provider.num_dates}")
print(f"Instruments: {provider.get_instruments()}")

In [ ]:
# Visualize the data
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(dates, aapl / aapl[0] * 100, label='AAPL', linewidth=1.5)
ax.plot(dates, msft / msft[0] * 100, label='MSFT', linewidth=1.5)
ax.plot(dates, googl / googl[0] * 100, label='GOOGL', linewidth=1.5)

ax.set_xlabel('Date')
ax.set_ylabel('Indexed Price (100 = Start)')
ax.set_title('Simulated Stock Prices')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Defining a Trading Strategy

A strategy is a function that takes market data, portfolio state, and context, and returns a list of orders.

In [ ]:
# Simple order class
@dataclass
class Order:
    instrument_id: str
    quantity: float

In [ ]:
# Strategy 1: Buy and Hold
def buy_and_hold(market, portfolio, context):
    """Buy equal weights of all stocks on day 1."""
    if context.step == 0:
        orders = []
        cash_per_stock = portfolio.cash / 3
        for ticker in ["AAPL", "MSFT", "GOOGL"]:
            price = market.get_price(ticker)
            shares = int(cash_per_stock / price)
            orders.append(Order(ticker, shares))
        return orders
    return []

In [ ]:
# Strategy 2: Moving Average Crossover
def ma_crossover(market, portfolio, context):
    """Buy when 10-day MA crosses above 50-day MA."""
    if "prices" not in context.user_data:
        context.user_data["prices"] = {ticker: [] for ticker in ["AAPL", "MSFT", "GOOGL"]}
    
    orders = []
    prices = context.user_data["prices"]
    
    for ticker in ["AAPL", "MSFT", "GOOGL"]:
        price = market.get_price(ticker)
        prices[ticker].append(price)
        
        if len(prices[ticker]) < 50:
            continue
        
        ma_fast = np.mean(prices[ticker][-10:])
        ma_slow = np.mean(prices[ticker][-50:])
        prev_fast = np.mean(prices[ticker][-11:-1])
        prev_slow = np.mean(prices[ticker][-51:-1])
        
        current_qty = portfolio.get_quantity(ticker)
        
        # Buy signal: fast crosses above slow
        if prev_fast <= prev_slow and ma_fast > ma_slow and current_qty == 0:
            shares = int(portfolio.cash * 0.3 / price)
            if shares > 0:
                orders.append(Order(ticker, shares))
        
        # Sell signal: fast crosses below slow
        elif prev_fast >= prev_slow and ma_fast < ma_slow and current_qty > 0:
            orders.append(Order(ticker, -current_qty))
    
    return orders

In [ ]:
# Strategy 3: Momentum
def momentum_strategy(market, portfolio, context):
    """Buy top performer over last 20 days."""
    if "prices" not in context.user_data:
        context.user_data["prices"] = {ticker: [] for ticker in ["AAPL", "MSFT", "GOOGL"]}
    
    prices = context.user_data["prices"]
    tickers = ["AAPL", "MSFT", "GOOGL"]
    
    for ticker in tickers:
        prices[ticker].append(market.get_price(ticker))
    
    if len(prices["AAPL"]) < 20:
        return []
    
    # Compute 20-day returns
    returns = {}
    for ticker in tickers:
        returns[ticker] = prices[ticker][-1] / prices[ticker][-20] - 1
    
    # Find best performer
    best = max(returns, key=returns.get)
    
    orders = []
    
    # Sell non-best holdings
    for ticker in tickers:
        qty = portfolio.get_quantity(ticker)
        if ticker != best and qty > 0:
            orders.append(Order(ticker, -qty))
    
    # Buy best if not held
    best_qty = portfolio.get_quantity(best)
    if best_qty == 0 and portfolio.cash > 1000:
        price = market.get_price(best)
        shares = int(portfolio.cash * 0.95 / price)
        if shares > 0:
            orders.append(Order(best, shares))
    
    return orders

## 3. Running Backtests

Now let's run all three strategies and compare their performance.

In [ ]:
# Configure the engine
config = BacktestConfig(
    transaction_cost=0.001,   # 10 bps
    slippage=0.0005,          # 5 bps
    risk_free_rate=0.02,      # 2% annual
    periods_per_year=252,
)

engine = BacktestEngine(config=config)

# Run all strategies
strategies = {
    "Buy & Hold": buy_and_hold,
    "MA Crossover": ma_crossover,
    "Momentum": momentum_strategy,
}

results = {}
for name, strategy in strategies.items():
    result = engine.run(
        strategy=strategy,
        data_provider=provider,
        initial_capital=100_000,
    )
    results[name] = result
    print(f"\n{name}:")
    print(f"  Final Value: ${result.final_value:,.2f}")
    print(f"  Total Return: {result.metrics.total_return:+.2%}")
    print(f"  Sharpe Ratio: {result.metrics.sharpe_ratio:.2f}")
    print(f"  Max Drawdown: {result.metrics.max_drawdown:.2%}")
    print(f"  Trades: {len(result.trades)}")

## 4. Visualizing Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {'Buy & Hold': '#1f77b4', 'MA Crossover': '#ff7f0e', 'Momentum': '#2ca02c'}

# 1. Portfolio Value
ax = axes[0, 0]
for name, result in results.items():
    ax.plot(result.dates, result.portfolio_values, label=name, color=colors[name], linewidth=1.5)
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value ($)')
ax.set_title('Portfolio Value Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Cumulative Returns
ax = axes[0, 1]
for name, result in results.items():
    ax.plot(result.dates, result.cumulative_returns * 100, label=name, color=colors[name], linewidth=1.5)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (%)')
ax.set_title('Cumulative Returns')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Drawdowns
ax = axes[1, 0]
for name, result in results.items():
    ax.fill_between(result.dates, -result.drawdown_series * 100, 0, 
                    alpha=0.3, label=name, color=colors[name])
    ax.plot(result.dates, -result.drawdown_series * 100, color=colors[name], linewidth=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown (%)')
ax.set_title('Underwater Curve (Drawdowns)')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Performance Summary
ax = axes[1, 1]
metrics_names = ['Total Return', 'Sharpe Ratio', 'Calmar Ratio', 'Win Rate']
x = np.arange(len(metrics_names))
width = 0.25

for i, (name, result) in enumerate(results.items()):
    values = [
        result.metrics.total_return * 100,
        result.metrics.sharpe_ratio,
        min(result.metrics.calmar_ratio, 5),  # Cap for display
        result.metrics.win_rate * 100,
    ]
    ax.bar(x + i * width, values, width, label=name, color=colors[name])

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_names)
ax.set_title('Performance Metrics Comparison')
ax.legend()
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Detailed Metrics Analysis

In [ ]:
# Create comparison table
import pandas as pd

comparison = []
for name, result in results.items():
    m = result.metrics
    comparison.append({
        'Strategy': name,
        'Total Return': f"{m.total_return:.2%}",
        'Ann. Return': f"{m.annualized_return:.2%}",
        'Ann. Vol': f"{m.annualized_volatility:.2%}",
        'Sharpe': f"{m.sharpe_ratio:.2f}",
        'Sortino': f"{m.sortino_ratio:.2f}",
        'Max DD': f"{m.max_drawdown:.2%}",
        'Calmar': f"{m.calmar_ratio:.2f}",
        'Win Rate': f"{m.win_rate:.1%}",
        'Trades': len(result.trades),
    })

df = pd.DataFrame(comparison)
print("Strategy Comparison:")
print(df.to_string(index=False))

In [ ]:
# Return distribution analysis
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, (name, result) in enumerate(results.items()):
    returns = result.returns[1:]  # Exclude first zero return
    
    ax = axes[i]
    ax.hist(returns * 100, bins=50, alpha=0.7, color=colors[name], edgecolor='black', linewidth=0.5)
    ax.axvline(np.mean(returns) * 100, color='red', linestyle='--', label=f'Mean: {np.mean(returns)*100:.2f}%')
    ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
    ax.set_xlabel('Daily Return (%)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{name} Return Distribution')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. P&L Attribution

Let's demonstrate how to attribute P&L to different Greek factors.

In [ ]:
from src.backtesting.attribution import (
    PnLBreakdown,
    PnLAttribution,
    attribute_pnl_to_greeks,
    aggregate_attribution,
)

# Example: Single day attribution
breakdown = attribute_pnl_to_greeks(
    pnl=5000,          # Total P&L
    delta=10000,       # Portfolio delta
    gamma=500,         # Portfolio gamma
    theta=-200,        # Portfolio theta (daily)
    vega=8000,         # Portfolio vega
    spot_move=0.5,     # Spot moved $0.50
    vol_move=0.005,    # Vol moved 0.5%
    dt=1/252,          # One day
)

print("Single Day P&L Attribution:")
print(breakdown)

In [ ]:
# Simulate multi-day attribution
np.random.seed(123)

attribution = PnLAttribution()

# Simulate 60 trading days
for i in range(60):
    dt = date(2024, 1, 1) + timedelta(days=i)
    
    # Simulated P&L and factor moves
    spot_move = np.random.normal(0, 2)
    vol_move = np.random.normal(0, 0.005)
    
    breakdown = attribute_pnl_to_greeks(
        pnl=10000 * spot_move + 8000 * vol_move - 200 + np.random.normal(0, 500),
        delta=10000,
        gamma=500,
        theta=-200,
        vega=8000,
        spot_move=spot_move,
        vol_move=vol_move,
        dt=1/252,
    )
    attribution.add(dt, breakdown)

print("Cumulative Attribution:")
print(attribution.cumulative)

In [ ]:
# Visualize attribution
arrays = attribution.to_arrays()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Cumulative P&L by factor
ax = axes[0]
factors = ['delta_pnl', 'gamma_pnl', 'theta_pnl', 'vega_pnl', 'residual']
labels = ['Delta', 'Gamma', 'Theta', 'Vega', 'Residual']
colors_attr = ['#1f77b4', '#ff7f0e', '#d62728', '#9467bd', '#7f7f7f']

cum_pnl = {f: np.cumsum(arrays[f]) for f in factors}
for factor, label, color in zip(factors, labels, colors_attr):
    ax.plot(attribution.dates, cum_pnl[factor], label=label, color=color, linewidth=1.5)

ax.plot(attribution.dates, np.cumsum(arrays['total_pnl']), 'k--', label='Total', linewidth=2)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative P&L ($)')
ax.set_title('Cumulative P&L Attribution')
ax.legend(loc='upper left')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.grid(True, alpha=0.3)

# Stacked area chart
ax = axes[1]
y_stack = np.vstack([arrays[f] for f in factors[:4]])  # Exclude residual
ax.stackplot(attribution.dates, y_stack, labels=labels[:4], colors=colors_attr[:4], alpha=0.7)
ax.bar(attribution.dates, arrays['residual'], alpha=0.5, color=colors_attr[4], label='Residual', width=1)
ax.set_xlabel('Date')
ax.set_ylabel('Daily P&L ($)')
ax.set_title('Daily P&L Breakdown')
ax.legend(loc='upper left')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Aggregate to weekly
weekly = aggregate_attribution(attribution, frequency="weekly")

print(f"Weekly periods: {len(weekly.dates)}")
print("\nWeekly Cumulative:")
print(weekly.cumulative)

## 7. Rolling Metrics Analysis

In [ ]:
# Compute rolling Sharpe for momentum strategy
result = results["Momentum"]
returns = result.returns[1:]  # Exclude first
window = 63  # ~3 months

rolling_sharpe = []
rolling_dates = []

for i in range(window, len(returns)):
    window_returns = returns[i-window:i]
    sharpe = compute_sharpe_ratio(window_returns, risk_free_rate=0.02)
    rolling_sharpe.append(sharpe)
    rolling_dates.append(result.dates[i])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(rolling_dates, rolling_sharpe, color='#2ca02c', linewidth=1.5)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Good (1.0)')
ax.axhline(y=2, color='blue', linestyle='--', alpha=0.5, label='Excellent (2.0)')
ax.fill_between(rolling_dates, rolling_sharpe, 0, 
                where=[s > 0 for s in rolling_sharpe], alpha=0.3, color='green')
ax.fill_between(rolling_dates, rolling_sharpe, 0, 
                where=[s <= 0 for s in rolling_sharpe], alpha=0.3, color='red')

ax.set_xlabel('Date')
ax.set_ylabel('Rolling Sharpe Ratio')
ax.set_title(f'Momentum Strategy: {window}-Day Rolling Sharpe Ratio')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Summary

This tutorial demonstrated:

1. **Data Preparation**: Creating price data and data providers
2. **Strategy Development**: Writing different trading strategies
3. **Backtest Execution**: Running backtests with realistic costs
4. **Performance Analysis**: Computing and comparing metrics
5. **P&L Attribution**: Breaking down returns by risk factor
6. **Visualization**: Creating professional charts

**Key Metrics Explained:**
- **Sharpe Ratio**: Risk-adjusted return (>1 is good, >2 is excellent)
- **Sortino Ratio**: Like Sharpe but only penalizes downside
- **Max Drawdown**: Worst peak-to-trough loss
- **Calmar Ratio**: Annual return / max drawdown
- **Win Rate**: Percentage of positive return periods

---

*See also:*
- [Technical Reference](../../reference/backtesting/backtesting_framework.md)
- [User Guide](../../guides/backtesting/backtesting_framework.md)